# Evaluation: Modelling Node
This notebook evaluates `modelling` predictions against benchmark labels and includes a concise first-iteration utility for reruns.

In [1]:
# Setup imports and project paths
import os
import sys
import json
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Resolve project root robustly from notebook location
ROOT = Path.cwd().resolve()
if not (ROOT / "nodes").exists():
    ROOT = ROOT.parent
if not (ROOT / "nodes").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PILOT = ROOT / "Pilot_Evaluation"
BENCHMARK_FILE = PILOT / "DATA_sample_10" / "Data Science Research Process (DSRP) Framework.csv"
RESULTS_FOLDER = PILOT / "pilot_study_results"
VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

print(f"Project root: {ROOT}")
print(f"Benchmark file exists: {BENCHMARK_FILE.exists()}")
print(f"Results folder exists: {RESULTS_FOLDER.exists()}")

Project root: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow
Benchmark file exists: True
Results folder exists: True


In [2]:
# Load benchmark and define modelling helpers
DIMENSIONS = [
    "Foundational Paradigm",
    "ML Learning Type",
    "ML Problem Type",
    "Deep Learning Used",
    "Specialised Paradigms",
]

PRED_KEY_MAP = {
    "Foundational Paradigm": "foundational_paradigm",
    "ML Learning Type": "ml_learning_type",
    "ML Problem Type": "ml_problem_type",
    "Deep Learning Used": "deep_learning_used",
    "Specialised Paradigms": "specialised_paradigms",
}

LIST_DIMENSIONS = {"ML Learning Type", "ML Problem Type", "Specialised Paradigms"}

FOUNDATIONAL_ALLOWED = [
    "Classical Statistical Modelling",
    "Machine Learning",
    "Mixed",
]

ML_LEARNING_ALLOWED = ["supervised", "unsupervised", "reinforcement", "not_applicable"]
ML_PROBLEM_ALLOWED = [
    "classification",
    "regression",
    "clustering",
    "dimensionality reduction",
    "reward optimisation",
    "not_applicable",
]
SPECIALISED_ALLOWED = [
    "Time-Series Analytics",
    "Natural Language Processing",
    "Spatial Analytics",
    "Network Analysis",
    "Emerging Paradigms",
]

def normalize_text(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    return text if text else None

def normalize_paper_id(value):
    text = normalize_text(value)
    return text.lower() if text else None

def normalize_bool(value):
    text = normalize_text(value)
    if not text:
        return False
    return text.lower() in {"true", "1", "yes", "y"}

def normalize_foundational(value):
    text = normalize_text(value)
    if not text:
        return None
    for canonical in FOUNDATIONAL_ALLOWED:
        if text.lower() == canonical.lower():
            return canonical
    return text

def _split_semicolon(value):
    text = normalize_text(value)
    if not text:
        return []
    return [p.strip() for p in text.split(";") if p and str(p).strip()]

def normalize_ml_learning(values):
    if not isinstance(values, list):
        values = _split_semicolon(values)
    mapping = {
        "supervised": "supervised",
        "supervised learning": "supervised",
        "unsupervised": "unsupervised",
        "unsupervised learning": "unsupervised",
        "reinforcement": "reinforcement",
        "reinforcement learning": "reinforcement",
        "not applicable": "not_applicable",
        "not_applicable": "not_applicable",
    }
    out = []
    for item in values:
        text = normalize_text(item)
        if not text:
            continue
        mapped = mapping.get(text.lower(), text.lower())
        if mapped not in out:
            out.append(mapped)
    return sorted(out)

def normalize_ml_problem(values):
    if not isinstance(values, list):
        values = _split_semicolon(values)
    mapping = {
        "classification": "classification",
        "regression": "regression",
        "clustering": "clustering",
        "dimensionality reduction": "dimensionality reduction",
        "dimensionality_reduction": "dimensionality reduction",
        "reward optimisation": "reward optimisation",
        "reward optimization": "reward optimisation",
        "not applicable": "not_applicable",
        "not_applicable": "not_applicable",
    }
    out = []
    for item in values:
        text = normalize_text(item)
        if not text:
            continue
        mapped = mapping.get(text.lower(), text.lower())
        if mapped not in out:
            out.append(mapped)
    return sorted(out)

def normalize_specialised(values):
    if not isinstance(values, list):
        values = _split_semicolon(values)
    out = []
    for item in values:
        text = normalize_text(item)
        if not text:
            continue
        matched = None
        for canonical in SPECIALISED_ALLOWED:
            if text.lower() == canonical.lower():
                matched = canonical
                break
        out.append(matched or text)
    return sorted(set(out))

def normalize_dimension(dim, value):
    if dim == "Foundational Paradigm":
        return normalize_foundational(value)
    if dim == "ML Learning Type":
        return normalize_ml_learning(value)
    if dim == "ML Problem Type":
        return normalize_ml_problem(value)
    if dim == "Deep Learning Used":
        return normalize_bool(value)
    if dim == "Specialised Paradigms":
        return normalize_specialised(value)
    return value

def format_dimension_value(dim, value):
    if dim in LIST_DIMENSIONS:
        return "; ".join(value) if value else "[]"
    if isinstance(value, bool):
        return str(value)
    return str(value) if value is not None else "(missing)"

def list_jaccard(gt_list, pred_list):
    gt_set = set(gt_list or [])
    pred_set = set(pred_list or [])
    if not gt_set and not pred_set:
        return 1.0
    return len(gt_set & pred_set) / len(gt_set | pred_set)

def get_nested(data, path, default=None):
    current = data
    for key in path.split("."):
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    return current

benchmark_df = pd.read_csv(BENCHMARK_FILE)
benchmark_df.columns = benchmark_df.columns.str.strip()

benchmark_col_map = {
    "Specialised Paradigms": "Specialised Paradigms (Skip if none)",
}
required_cols = ["Paper ID", "Gatekeeper", "Foundational Paradigm", "ML Learning Type", "ML Problem Type", "Deep Learning Used", benchmark_col_map["Specialised Paradigms"]]
missing = [c for c in required_cols if c not in benchmark_df.columns]
if missing:
    raise ValueError(f"Missing required benchmark columns: {missing}")

benchmark_eval = benchmark_df[required_cols].copy()
benchmark_eval = benchmark_eval.rename(columns={benchmark_col_map["Specialised Paradigms"]: "Specialised Paradigms"})
benchmark_eval["Paper ID"] = benchmark_eval["Paper ID"].apply(normalize_paper_id)
benchmark_eval["Gatekeeper"] = benchmark_eval["Gatekeeper"].astype(str).str.strip().str.lower()

for dim in DIMENSIONS:
    benchmark_eval[dim] = benchmark_eval[dim].apply(lambda v: normalize_dimension(dim, v))

benchmark_eval = benchmark_eval[benchmark_eval["Gatekeeper"] == "include"].copy()
benchmark_eval = benchmark_eval.dropna(subset=["Paper ID"]).reset_index(drop=True)

print(f"Included benchmark rows: {len(benchmark_eval)}")
preview_cols = ["Paper ID", *DIMENSIONS]
print(benchmark_eval[preview_cols].head().to_string(index=False))

Included benchmark rows: 7
 Paper ID           Foundational Paradigm           ML Learning Type              ML Problem Type  Deep Learning Used                                Specialised Paradigms
 2018 - 8                           Mixed               [supervised] [classification, regression]               False                                                   []
2019 - 33 Classical Statistical Modelling                         []                           []               False                                                   []
 2020 - 8                Machine Learning             [unsupervised]                 [clustering]               False      [Natural Language Processing, Network Analysis]
2021 - 28                           Mixed               [supervised]                 [regression]               False                                                   []
2021 - 56                           Mixed [supervised, unsupervised]     [clustering, regression]               False 

In [3]:
# Load existing agent outputs and build comparison table
def load_agent_results(results_folder: Path):
    agent_data = {}
    for paper_dir in sorted(results_folder.glob("*/")):
        if not paper_dir.is_dir():
            continue
        results_file = paper_dir / "aggregated" / "results.json"
        if not results_file.exists():
            continue
        with open(results_file, "r", encoding="utf-8") as f:
            agent_data[paper_dir.name.strip().lower()] = json.load(f)
    return agent_data

def extract_modelling_prediction(modelling_obj, dim):
    key = PRED_KEY_MAP[dim]
    return normalize_dimension(dim, modelling_obj.get(key))

agent_data = load_agent_results(RESULTS_FOLDER)
print(f"Loaded aggregated outputs for {len(agent_data)} papers")

comparison_rows = []
missing_agent_results = []

for _, row in benchmark_eval.iterrows():
    paper_id = row["Paper ID"]
    if paper_id not in agent_data:
        missing_agent_results.append(paper_id)
        continue

    outputs = agent_data[paper_id]
    modelling_obj = get_nested(outputs, "dsrp_outputs.modelling", default={})

    record = {"Paper ID": paper_id}
    dim_match_flags = []
    for dim in DIMENSIONS:
        gt_value = normalize_dimension(dim, row[dim])
        pred_value = extract_modelling_prediction(modelling_obj, dim)
        is_match = gt_value == pred_value
        dim_match_flags.append(is_match)

        record[f"GT_{dim}"] = format_dimension_value(dim, gt_value)
        record[f"Agent_{dim}"] = format_dimension_value(dim, pred_value)
        record[f"Match_{dim}"] = "MATCH" if is_match else "MISMATCH"

        if dim in LIST_DIMENSIONS:
            record[f"Jaccard_{dim}"] = list_jaccard(gt_value, pred_value)

    record["Match_All_5"] = "MATCH" if all(dim_match_flags) else "MISMATCH"
    comparison_rows.append(record)

comparison_df = pd.DataFrame(comparison_rows)
print(f"Comparable rows: {len(comparison_df)}")
if missing_agent_results:
    print(f"Missing agent outputs for {len(missing_agent_results)} papers: {', '.join(missing_agent_results)}")

if len(comparison_df):
    show_cols = ["Paper ID", *[f"Match_{d}" for d in DIMENSIONS], "Match_All_5"]
    print(comparison_df[show_cols].to_string(index=False))
else:
    print("No comparable rows found.")

Loaded aggregated outputs for 10 papers
Comparable rows: 7
 Paper ID Match_Foundational Paradigm Match_ML Learning Type Match_ML Problem Type Match_Deep Learning Used Match_Specialised Paradigms Match_All_5
 2018 - 8                       MATCH                  MATCH                 MATCH                    MATCH                       MATCH       MATCH
2019 - 33                       MATCH                  MATCH                 MATCH                    MATCH                       MATCH       MATCH
 2020 - 8                       MATCH                  MATCH                 MATCH                    MATCH                    MISMATCH    MISMATCH
2021 - 28                       MATCH                  MATCH                 MATCH                    MATCH                       MATCH       MATCH
2021 - 56                       MATCH               MISMATCH              MISMATCH                    MATCH                    MISMATCH    MISMATCH
2023 - 58                    MISMATCH               M

In [4]:
# Dimension-wise metrics and overall agreement
if len(comparison_df) == 0:
    raise ValueError("No rows available for evaluation.")

metrics_rows = []
for dim in DIMENSIONS:
    y_true = comparison_df[f"GT_{dim}"].tolist()
    y_pred = comparison_df[f"Agent_{dim}"].tolist()

    acc = accuracy_score(y_true, y_pred)
    precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

    row = {
        "dimension": dim,
        "samples": len(y_true),
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
    }

    if dim in LIST_DIMENSIONS:
        row["avg_jaccard"] = comparison_df[f"Jaccard_{dim}"].mean()

    metrics_rows.append(row)

    print("=" * 90)
    print(f"DIMENSION: {dim}")
    print("=" * 90)
    print(f"accuracy............ {acc:.4f}")
    print(f"precision_macro..... {precision_macro:.4f}")
    print(f"recall_macro........ {recall_macro:.4f}")
    print(f"f1_macro............ {f1_macro:.4f}")
    if dim in LIST_DIMENSIONS:
        print(f"avg_jaccard......... {comparison_df[f'Jaccard_{dim}'].mean():.4f}")

    report = classification_report(
        y_true,
        y_pred,
        zero_division=0,
        output_dict=True,
    )
    report_df = pd.DataFrame(report).T
    print("\nPer-label report:")
    print(report_df.to_string())

metrics_df = pd.DataFrame(metrics_rows)
overall_match_count = int((comparison_df["Match_All_5"] == "MATCH").sum())
overall_total = len(comparison_df)
print("\nSUMMARY TABLE")
print(metrics_df.to_string(index=False))
print(f"\nOverall exact agreement across all 5 modelling dimensions: {overall_match_count}/{overall_total} ({100 * overall_match_count / overall_total:.1f}%)")

DIMENSION: Foundational Paradigm
accuracy............ 0.8571
precision_macro..... 0.9167
recall_macro........ 0.8889
f1_macro............ 0.8857

Per-label report:
                                 precision    recall  f1-score   support
Classical Statistical Modelling   1.000000  1.000000  1.000000  1.000000
Machine Learning                  1.000000  0.666667  0.800000  3.000000
Mixed                             0.750000  1.000000  0.857143  3.000000
accuracy                          0.857143  0.857143  0.857143  0.857143
macro avg                         0.916667  0.888889  0.885714  7.000000
weighted avg                      0.892857  0.857143  0.853061  7.000000
DIMENSION: ML Learning Type
accuracy............ 0.5714
precision_macro..... 0.3611
recall_macro........ 0.4167
f1_macro............ 0.3556
avg_jaccard......... 0.6429

Per-label report:
                             precision    recall  f1-score   support
[]                            0.500000  1.000000  0.666667  1.000000


In [5]:
# Detailed mismatch diagnostics
mismatch_df = comparison_df[comparison_df["Match_All_5"] == "MISMATCH"].copy()
print(f"Total mismatched papers: {len(mismatch_df)}")
if len(mismatch_df):
    show_cols = [
        "Paper ID",
        *[f"GT_{d}" for d in DIMENSIONS],
        *[f"Agent_{d}" for d in DIMENSIONS],
        *[f"Match_{d}" for d in DIMENSIONS],
        "Match_All_5",
    ]
    print(mismatch_df[show_cols].to_string(index=False))
else:
    print("No mismatches found.")

Total mismatched papers: 4
 Paper ID GT_Foundational Paradigm         GT_ML Learning Type     GT_ML Problem Type GT_Deep Learning Used                                           GT_Specialised Paradigms Agent_Foundational Paradigm Agent_ML Learning Type Agent_ML Problem Type Agent_Deep Learning Used Agent_Specialised Paradigms Match_Foundational Paradigm Match_ML Learning Type Match_ML Problem Type Match_Deep Learning Used Match_Specialised Paradigms Match_All_5
 2020 - 8         Machine Learning                unsupervised             clustering                 False                      Natural Language Processing; Network Analysis            Machine Learning           unsupervised            clustering                    False Natural Language Processing                       MATCH                  MATCH                 MATCH                    MATCH                    MISMATCH    MISMATCH
2021 - 56                    Mixed    supervised; unsupervised clustering; regression          

## Iteration Utility
Use the function below to re-run `modelling_node` on selected papers while keeping iteration code out of analysis cells.

In [6]:
import importlib
import textwrap
import nodes.modelling_node as modelling_module
from utils.dsrp_state import DSRPState

def _format_bibliography_text(bibliography):
    entries = bibliography or []
    if not entries:
        return ["(no bibliography)"]
    lines = []
    for i, b in enumerate(entries, start=1):
        sec = b.get("section", "")
        page = b.get("page", "")
        quote = b.get("direct_quote", "")
        lines.append(f"[{i}] page={page} | section={sec}")
        if quote:
            wrapped = textwrap.wrap(str(quote), width=110)
            if wrapped:
                lines.extend([f"    quote: {wrapped[0]}"] + [f"           {w}" for w in wrapped[1:]])
    return lines

def _extract_reasoning_payload(modelling_obj):
    return {
        "overall_validated_reasoning": modelling_obj.get("validated_reasoning", ""),
        "audit_commentary": modelling_obj.get("audit_commentary", ""),
        "confidence": modelling_obj.get("confidence", None),
        "validated_bibliography_count": len(modelling_obj.get("validated_bibliography", []) or []),
        "validated_bibliography_lines": _format_bibliography_text(modelling_obj.get("validated_bibliography", [])),
    }

def _print_iteration_pretty(record, show_bibliography=True):
    print("\n" + "=" * 100)
    print(f"PAPER: {record.get('Paper ID', '(missing)')} | MODEL: {record.get('Model', '(missing)')}")
    print("=" * 100)
    if record.get("Error"):
        print(f"Error: {record['Error']}")
        return

    for dim in DIMENSIONS:
        print(f"\n{dim}")
        print(f"  GT    : {record.get(f'GT_{dim}', '(missing)')}")
        print(f"  Agent : {record.get(f'New_Agent_{dim}', '(missing)')}")
        print(f"  Match : {record.get(f'Match_{dim}', '(missing)')}")

    print("\nOverall all-5-dimensions match:", record.get("Match_All_5", "(missing)"))
    print("Confidence:", record.get("confidence", "(missing)"))
    print("Overall validated reasoning:")
    reasoning = str(record.get("overall_validated_reasoning", "") or "(none)")
    for line in textwrap.wrap(reasoning, width=110):
        print(f"  {line}")
    print("Audit commentary:")
    commentary = str(record.get("audit_commentary", "") or "(none)")
    for line in textwrap.wrap(commentary, width=110):
        print(f"  {line}")
    if show_bibliography:
        print("Bibliography:")
        for line in record.get("validated_bibliography_lines", ["(no bibliography)"]):
            print(f"  {line}")

def run_modelling_iteration(
    paper_ids,
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_reasoning_in_logs=False,
    log_bibliography=False,
    ):
    """Run modelling_node over selected papers and return a compact comparison table."""
    importlib.reload(modelling_module)
    modelling_node = modelling_module.modelling_node

    benchmark_lookup = benchmark_eval.set_index("Paper ID")
    rows = []

    original_dir = Path.cwd().resolve()
    os.chdir(ROOT)
    try:
        for paper_id in paper_ids:
            pid = normalize_paper_id(paper_id)
            if not pid:
                continue
            if pid not in benchmark_lookup.index:
                rows.append({"Paper ID": pid, "Error": "Paper ID not found in benchmark include set"})
                continue

            print(f"Re-running modelling node for {pid} with {llm_model}...")
            state = DSRPState(
                paper_id=pid,
                gatekeeper={},
                dsrp_outputs={},
                collection_name=COLLECTION_NAME,
                persist_directory=str(VECTOR_DB_PATH),
                embedding_model=EMBEDDING_MODEL,
                llm_model=llm_model,
            )

            try:
                out = modelling_node(state)
                modelling_obj = get_nested(out, "dsrp_outputs.modelling", default={})

                record = {"Paper ID": pid, "Model": llm_model}
                match_flags = []
                for dim in DIMENSIONS:
                    gt_value = normalize_dimension(dim, benchmark_lookup.loc[pid, dim])
                    pred_value = normalize_dimension(dim, modelling_obj.get(PRED_KEY_MAP[dim]))
                    dim_match = gt_value == pred_value
                    match_flags.append(dim_match)

                    record[f"GT_{dim}"] = format_dimension_value(dim, gt_value)
                    record[f"New_Agent_{dim}"] = format_dimension_value(dim, pred_value)
                    record[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

                if include_reasoning:
                    record.update(_extract_reasoning_payload(modelling_obj))

                record["Match_All_5"] = "MATCH" if all(match_flags) else "MISMATCH"
                rows.append(record)

                if verbose:
                    label_line = " | ".join(
                        f"{dim}: {record.get(f'New_Agent_{dim}', '(missing)')}"
                        for dim in DIMENSIONS
                    )
                    print(f"  Assigned labels -> {label_line}")
                    print(f"  Match_All_5: {record['Match_All_5']}")

                if verbose and include_reasoning and show_reasoning_in_logs:
                    _print_iteration_pretty(record, show_bibliography=log_bibliography)

            except Exception as e:
                err_row = {"Paper ID": pid, "Model": llm_model, "Error": str(e)}
                rows.append(err_row)
                if verbose:
                    print(f"  Error: {e}")
    finally:
        os.chdir(original_dir)

    retest_df = pd.DataFrame(rows)
    if len(retest_df):
        print("\n" + "-" * 110)
        print("RE-TEST RESULTS (compact summary)")
        print("-" * 110)
        show_cols = [
            "Paper ID",
            "Model",
            *[col for d in DIMENSIONS for col in (f"GT_{d}", f"New_Agent_{d}", f"Match_{d}")],
            "Match_All_5",
        ]
        cols = [c for c in show_cols if c in retest_df.columns]
        if cols:
            print(retest_df[cols].to_string(index=False))
        else:
            print(retest_df.to_string(index=False))
    return retest_df

def show_iteration_reasoning(retest_df, paper_id, show_bibliography=True):
    pid = normalize_paper_id(paper_id)
    if "Paper ID" not in retest_df.columns:
        raise ValueError("retest_df does not contain 'Paper ID'.")

    rows = retest_df[retest_df["Paper ID"] == pid]
    if len(rows) == 0:
        raise ValueError(f"Paper ID not found in retest_df: {pid}")

    row = rows.iloc[0].to_dict()
    _print_iteration_pretty(row, show_bibliography=show_bibliography)

## Iteration 1
Run a single first iteration on mismatched papers from the baseline comparison.

In [18]:
# Iteration 1: baseline rerun on papers mismatched in baseline outputs
papers_to_rerun = comparison_df[comparison_df["Match_All_5"] == "MISMATCH"]["Paper ID"].tolist()
print(f"Papers selected for Iteration 1 rerun: {len(papers_to_rerun)}")

retest_df = run_modelling_iteration(
    paper_ids=papers_to_rerun,
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_reasoning_in_logs=False,
    log_bibliography=False,
)

Papers selected for Iteration 1 rerun: 4
Re-running modelling node for 2020 - 8 with gpt-4o-mini...
  Assigned labels -> Foundational Paradigm: Machine Learning | ML Learning Type: unsupervised | ML Problem Type: clustering | Deep Learning Used: False | Specialised Paradigms: Natural Language Processing
  Match_All_5: MISMATCH
Re-running modelling node for 2021 - 56 with gpt-4o-mini...
  Assigned labels -> Foundational Paradigm: Mixed | ML Learning Type: supervised | ML Problem Type: regression | Deep Learning Used: False | Specialised Paradigms: Natural Language Processing; Time-Series Analytics
  Match_All_5: MISMATCH
Re-running modelling node for 2023 - 58 with gpt-4o-mini...
  Assigned labels -> Foundational Paradigm: Mixed | ML Learning Type: unsupervised | ML Problem Type: clustering | Deep Learning Used: False | Specialised Paradigms: Emerging Paradigms; Natural Language Processing; Spatial Analytics
  Match_All_5: MISMATCH
Re-running modelling node for 2024 - 99 with gpt-4o-min

# Inspection

In [7]:
# Optional: inspect detailed reasoning for one rerun paper
if "retest_df" in globals() and len(retest_df):
    example_pid = "2023 - 58"
    show_iteration_reasoning(retest_df, example_pid, show_bibliography=True)
else:
    print("retest_df not found. Run the Iteration 1 cell first.")

retest_df not found. Run the Iteration 1 cell first.


In [10]:
run_modelling_iteration(
    paper_ids= ['2021 - 56'], # Enter Paper ID HERE. 
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_reasoning_in_logs=True,
    log_bibliography=True
)

Re-running modelling node for 2021 - 56 with gpt-4o-mini...
  Assigned labels -> Foundational Paradigm: Mixed | ML Learning Type: supervised; unsupervised | ML Problem Type: clustering; regression | Deep Learning Used: False | Specialised Paradigms: Natural Language Processing; Time-Series Analytics
  Match_All_5: MATCH

PAPER: 2021 - 56 | MODEL: gpt-4o-mini

Foundational Paradigm
  GT    : Mixed
  Agent : Mixed
  Match : MATCH

ML Learning Type
  GT    : supervised; unsupervised
  Agent : supervised; unsupervised
  Match : MATCH

ML Problem Type
  GT    : clustering; regression
  Agent : clustering; regression
  Match : MATCH

Deep Learning Used
  GT    : False
  Agent : False
  Match : MATCH

Specialised Paradigms
  GT    : Natural Language Processing; Time-Series Analytics
  Agent : Natural Language Processing; Time-Series Analytics
  Match : MATCH

Overall all-5-dimensions match: MATCH
Confidence: 0.9
Overall validated reasoning:
  The study employs both classical statistical model

,Paper ID,Model,GT_Foundational Paradigm,New_Agent_Foundational Paradigm,Match_Foundational Paradigm,GT_ML Learning Type,New_Agent_ML Learning Type,Match_ML Learning Type,GT_ML Problem Type,New_Agent_ML Problem Type,...,Match_Deep Learning Used,GT_Specialised Paradigms,New_Agent_Specialised Paradigms,Match_Specialised Paradigms,overall_validated_reasoning,audit_commentary,confidence,validated_bibliography_count,validated_bibliography_lines,Match_All_5
0,2021 - 56,gpt-4o-mini,Mixed,Mixed,MATCH,supervised; unsupervised,supervised; unsupervised,MATCH,clustering; regression,clustering; regression,...,MATCH,Natural Language Processing; Time-Series Analy...,Natural Language Processing; Time-Series Analy...,MATCH,The study employs both classical statistical m...,,0.9,10,"[[1] page=1 | section=Abstract, quote: Thi...",MATCH


# Re-run on All Included Papers

In [8]:
all_papers = comparison_df["Paper ID"].tolist()
print(f'length of all papers: {len(all_papers)}')
all_papers

length of all papers: 7


['2018 - 8',
 '2019 - 33',
 '2020 - 8',
 '2021 - 28',
 '2021 - 56',
 '2023 - 58',
 '2024 - 99']

In [10]:
full_retest = run_modelling_iteration(
    paper_ids= all_papers,  
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_reasoning_in_logs=True,
    log_bibliography=True
)
full_retest

Re-running modelling node for 2018 - 8 with gpt-4o-mini...


C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\utils\paper_retriever.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self._vectorstore = Chroma(


  Assigned labels -> Foundational Paradigm: Mixed | ML Learning Type: supervised | ML Problem Type: regression | Deep Learning Used: False | Specialised Paradigms: []
  Match_All_5: MISMATCH

PAPER: 2018 - 8 | MODEL: gpt-4o-mini

Foundational Paradigm
  GT    : Mixed
  Agent : Mixed
  Match : MATCH

ML Learning Type
  GT    : supervised
  Agent : supervised
  Match : MATCH

ML Problem Type
  GT    : classification; regression
  Agent : regression
  Match : MISMATCH

Deep Learning Used
  GT    : False
  Agent : False
  Match : MATCH

Specialised Paradigms
  GT    : []
  Agent : []
  Match : MATCH

Overall all-5-dimensions match: MISMATCH
Confidence: 1.0
Overall validated reasoning:
  The foundational paradigm is classified as Mixed, as the study employs both classical statistical models
  (Lasso regression) and machine learning models (Random Forest) in a two-stage modeling approach. The evidence
  supports the classification of both methodologies under supervised learning, as they are 

,Paper ID,Model,GT_Foundational Paradigm,New_Agent_Foundational Paradigm,Match_Foundational Paradigm,GT_ML Learning Type,New_Agent_ML Learning Type,Match_ML Learning Type,GT_ML Problem Type,New_Agent_ML Problem Type,...,Match_Deep Learning Used,GT_Specialised Paradigms,New_Agent_Specialised Paradigms,Match_Specialised Paradigms,overall_validated_reasoning,audit_commentary,confidence,validated_bibliography_count,validated_bibliography_lines,Match_All_5
0,2018 - 8,gpt-4o-mini,Mixed,Mixed,MATCH,supervised,supervised,MATCH,classification; regression,regression,...,MATCH,[],[],MATCH,The foundational paradigm is classified as Mix...,No inconsistencies were found across the layer...,1.0,10,"[[1] page=4 | section=3. Methodology, quot...",MISMATCH
1,2019 - 33,gpt-4o-mini,Classical Statistical Modelling,Classical Statistical Modelling,MATCH,[],[],MATCH,[],[],...,MATCH,[],[],MATCH,The study employs conventional statistical tes...,No inconsistencies were found across the layer...,0.9,4,"[[1] page=2 | section=Methodology, quote: ...",MATCH
2,2020 - 8,gpt-4o-mini,Machine Learning,Machine Learning,MATCH,unsupervised,unsupervised,MATCH,clustering; dimensionality reduction,clustering,...,MATCH,Natural Language Processing; Network Analysis,Natural Language Processing; Network Analysis,MATCH,The study employs Latent Dirichlet Allocation ...,,1.0,8,"[[1] page=6 | section=JHTT, quote: We then...",MISMATCH
3,2021 - 28,gpt-4o-mini,Mixed,Mixed,MATCH,supervised,supervised,MATCH,regression,regression,...,MATCH,[],[],MATCH,The foundational paradigm is classified as Mix...,The classifications are logically consistent a...,0.8,3,"[[1] page=4 | section=4. Methods, quote: W...",MATCH
4,2021 - 56,gpt-4o-mini,Mixed,Mixed,MATCH,supervised; unsupervised,supervised; unsupervised,MATCH,clustering; regression,clustering; regression,...,MATCH,Natural Language Processing; Time-Series Analy...,Natural Language Processing; Time-Series Analy...,MATCH,The foundational paradigm is classified as Mix...,,1.0,12,"[[1] page=1 | section=Abstract, quote: thi...",MATCH
5,2023 - 58,gpt-4o-mini,Machine Learning,Machine Learning,MATCH,unsupervised,unsupervised,MATCH,clustering,clustering,...,MATCH,Emerging Paradigms; Natural Language Processin...,Emerging Paradigms; Natural Language Processin...,MATCH,The study employs Latent Dirichlet Allocation ...,,0.9,10,[[1] page=6 | section=3.2. Topic modelling (LD...,MATCH
6,2024 - 99,gpt-4o-mini,Machine Learning,Machine Learning,MATCH,optimization-based learning,optimization-based learning,MATCH,reward optimisation,reward optimisation,...,MATCH,[],[],MATCH,The evidence indicates that the OptiPres Model...,No specialised paradigms were identified in th...,0.8,3,[[1] page=6 | section=OptiPres Model Objective...,MATCH


#### Iteration 2: On Full Set

In [10]:
full_retest = run_modelling_iteration(
    paper_ids= all_papers,  
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_reasoning_in_logs=True,
    log_bibliography=True
)
full_retest

Re-running modelling node for 2018 - 8 with gpt-4o-mini...


C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\utils\paper_retriever.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self._vectorstore = Chroma(


  Assigned labels -> Foundational Paradigm: Mixed | ML Learning Type: supervised | ML Problem Type: classification; regression | Deep Learning Used: False | Specialised Paradigms: []
  Match_All_5: MATCH

PAPER: 2018 - 8 | MODEL: gpt-4o-mini

Foundational Paradigm
  GT    : Mixed
  Agent : Mixed
  Match : MATCH

ML Learning Type
  GT    : supervised
  Agent : supervised
  Match : MATCH

ML Problem Type
  GT    : classification; regression
  Agent : classification; regression
  Match : MATCH

Deep Learning Used
  GT    : False
  Agent : False
  Match : MATCH

Specialised Paradigms
  GT    : []
  Agent : []
  Match : MATCH

Overall all-5-dimensions match: MATCH
Confidence: 0.8
Overall validated reasoning:
  The foundational paradigm is classified as 'Mixed' since the study employs both classical statistical models
  (Lasso regression and Logit model) and machine learning models (Random Forest). The presence of a
  'Classification Step' indicates the use of logistic regression for predict

,Paper ID,Model,GT_Foundational Paradigm,New_Agent_Foundational Paradigm,Match_Foundational Paradigm,GT_ML Learning Type,New_Agent_ML Learning Type,Match_ML Learning Type,GT_ML Problem Type,New_Agent_ML Problem Type,...,Match_Deep Learning Used,GT_Specialised Paradigms,New_Agent_Specialised Paradigms,Match_Specialised Paradigms,overall_validated_reasoning,audit_commentary,confidence,validated_bibliography_count,validated_bibliography_lines,Match_All_5
0,2018 - 8,gpt-4o-mini,Mixed,Mixed,MATCH,supervised,supervised,MATCH,classification; regression,classification; regression,...,MATCH,[],[],MATCH,The foundational paradigm is classified as 'Mi...,"The study's methodologies are well-supported, ...",0.8,7,"[[1] page=4 | section=3. Methodology, quot...",MATCH
1,2019 - 33,gpt-4o-mini,Classical Statistical Modelling,Classical Statistical Modelling,MATCH,[],[],MATCH,[],[],...,MATCH,[],[],MATCH,The foundational paradigm is validated as Clas...,The foundational paradigm is consistent with t...,0.9,3,[[1] page=2 | section=2. Mobile online reviews...,MATCH
2,2020 - 8,gpt-4o-mini,Machine Learning,Machine Learning,MATCH,unsupervised,unsupervised,MATCH,clustering,clustering,...,MATCH,Natural Language Processing; Network Analysis,Natural Language Processing; Network Analysis,MATCH,The foundational paradigm is validated as 'Mac...,The configurations across layers are consisten...,1.0,9,"[[1] page=6 | section=JHTT, quote: We then...",MATCH
3,2021 - 28,gpt-4o-mini,Mixed,Mixed,MATCH,supervised,supervised,MATCH,regression,[],...,MATCH,[],[],MATCH,The study employs both classical statistical m...,The foundational paradigm is validated as mixe...,0.9,3,"[[1] page=4 | section=Methods, quote: We e...",MISMATCH
4,2021 - 56,gpt-4o-mini,Mixed,Mixed,MATCH,supervised; unsupervised,supervised; unsupervised,MATCH,clustering; regression,clustering; regression,...,MATCH,Natural Language Processing; Time-Series Analy...,Natural Language Processing; Time-Series Analy...,MATCH,The study employs both classical statistical m...,,1.0,15,"[[1] page=1 | section=Abstract, quote: To ...",MATCH
5,2023 - 58,gpt-4o-mini,Machine Learning,Machine Learning,MATCH,unsupervised,unsupervised,MATCH,clustering,clustering,...,MATCH,Emerging Paradigms; Natural Language Processin...,Emerging Paradigms; Natural Language Processin...,MATCH,The foundational paradigm is classified as Mac...,,1.0,6,[[1] page=6 | section=3.2. Topic modelling (LD...,MATCH
6,2024 - 99,gpt-4o-mini,Machine Learning,Machine Learning,MATCH,optimization-based learning,optimization-based learning,MATCH,reward optimisation,reward optimisation,...,MATCH,[],[],MATCH,The foundational paradigm is validated as Mach...,The foundational model's classification is con...,0.8,4,[[1] page=6 | section=OptiPres Model Objective...,MATCH


#### Iteration 3: On Full Set (re-testing)

In [13]:
full_retest = run_modelling_iteration(
    paper_ids= all_papers,  
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_reasoning_in_logs=True,
    log_bibliography=True
)
full_retest

Re-running modelling node for 2018 - 8 with gpt-4o-mini...
  Assigned labels -> Foundational Paradigm: Mixed | ML Learning Type: supervised | ML Problem Type: classification; regression | Deep Learning Used: False | Specialised Paradigms: []
  Match_All_5: MATCH

PAPER: 2018 - 8 | MODEL: gpt-4o-mini

Foundational Paradigm
  GT    : Mixed
  Agent : Mixed
  Match : MATCH

ML Learning Type
  GT    : supervised
  Agent : supervised
  Match : MATCH

ML Problem Type
  GT    : classification; regression
  Agent : classification; regression
  Match : MATCH

Deep Learning Used
  GT    : False
  Agent : False
  Match : MATCH

Specialised Paradigms
  GT    : []
  Agent : []
  Match : MATCH

Overall all-5-dimensions match: MATCH
Confidence: 0.9
Overall validated reasoning:
  The study employs both classical statistical models and machine learning models. It utilizes a Logit model
  with Lasso penalty for classification, which indicates a classical statistical approach focused on inference
  and pr

,Paper ID,Model,GT_Foundational Paradigm,New_Agent_Foundational Paradigm,Match_Foundational Paradigm,GT_ML Learning Type,New_Agent_ML Learning Type,Match_ML Learning Type,GT_ML Problem Type,New_Agent_ML Problem Type,...,Match_Deep Learning Used,GT_Specialised Paradigms,New_Agent_Specialised Paradigms,Match_Specialised Paradigms,overall_validated_reasoning,audit_commentary,confidence,validated_bibliography_count,validated_bibliography_lines,Match_All_5
0,2018 - 8,gpt-4o-mini,Mixed,Mixed,MATCH,supervised,supervised,MATCH,classification; regression,classification; regression,...,MATCH,[],[],MATCH,The study employs both classical statistical m...,,0.9,5,"[[1] page=4 | section=3. Methodology, quot...",MATCH
1,2019 - 33,gpt-4o-mini,Classical Statistical Modelling,Classical Statistical Modelling,MATCH,[],[],MATCH,[],[],...,MATCH,[],[],MATCH,The study employs conventional statistical tes...,Validated foundational paradigm as Classical S...,0.9,3,"[[1] page=2 | section=Methodology, quote: ...",MATCH
2,2020 - 8,gpt-4o-mini,Machine Learning,Machine Learning,MATCH,unsupervised,unsupervised,MATCH,clustering,clustering,...,MATCH,Natural Language Processing; Network Analysis,Natural Language Processing; Network Analysis,MATCH,The study employs topic modeling techniques su...,,0.9,6,"[[1] page=6 | section=JHTT, quote: We then...",MATCH
3,2021 - 28,gpt-4o-mini,Mixed,Mixed,MATCH,supervised,supervised,MATCH,regression,[],...,MATCH,[],[],MATCH,The foundational paradigm is classified as 'Mi...,The foundational paradigm classification as 'M...,0.9,3,"[[1] page=4 | section=4. Methods, quote: W...",MISMATCH
4,2021 - 56,gpt-4o-mini,Mixed,Mixed,MATCH,supervised; unsupervised,supervised; unsupervised,MATCH,clustering; regression,clustering; regression,...,MATCH,Natural Language Processing; Time-Series Analy...,Natural Language Processing; Time-Series Analy...,MATCH,The study employs both classical statistical m...,,1.0,10,"[[1] page=2 | section=, quote: this study ...",MATCH
5,2023 - 58,gpt-4o-mini,Machine Learning,Machine Learning,MATCH,unsupervised,unsupervised,MATCH,clustering,clustering,...,MATCH,Emerging Paradigms; Natural Language Processin...,Emerging Paradigms; Natural Language Processin...,MATCH,The foundational paradigm is consistent with t...,,0.9,7,[[1] page=6 | section=3.2. Topic modelling (LD...,MATCH
6,2024 - 99,gpt-4o-mini,Machine Learning,Machine Learning,MATCH,optimization-based learning,optimization-based learning,MATCH,reward optimisation,reward optimisation,...,MATCH,[],[],MATCH,The study employs simulated annealing as part ...,All classifications and reasoning are consiste...,1.0,2,[[1] page=6 | section=OptiPres Model Objective...,MATCH


#### Iteration 4: On Full Set (After regression label fix)

In [17]:
full_retest = run_modelling_iteration(
    paper_ids= all_papers,  
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_reasoning_in_logs=True,
    log_bibliography=True
)
full_retest

Re-running modelling node for 2018 - 8 with gpt-4o-mini...
  Assigned labels -> Foundational Paradigm: Mixed | ML Learning Type: supervised | ML Problem Type: classification; regression | Deep Learning Used: False | Specialised Paradigms: []
  Match_All_5: MATCH

PAPER: 2018 - 8 | MODEL: gpt-4o-mini

Foundational Paradigm
  GT    : Mixed
  Agent : Mixed
  Match : MATCH

ML Learning Type
  GT    : supervised
  Agent : supervised
  Match : MATCH

ML Problem Type
  GT    : classification; regression
  Agent : classification; regression
  Match : MATCH

Deep Learning Used
  GT    : False
  Agent : False
  Match : MATCH

Specialised Paradigms
  GT    : []
  Agent : []
  Match : MATCH

Overall all-5-dimensions match: MATCH
Confidence: 0.8
Overall validated reasoning:
  The foundational paradigm is classified as Mixed, as the study employs both classical statistical models
  (Logit with Lasso and regression with Lasso) and machine learning models (Random Forest). The learning type is
  valida

,Paper ID,Model,GT_Foundational Paradigm,New_Agent_Foundational Paradigm,Match_Foundational Paradigm,GT_ML Learning Type,New_Agent_ML Learning Type,Match_ML Learning Type,GT_ML Problem Type,New_Agent_ML Problem Type,...,Match_Deep Learning Used,GT_Specialised Paradigms,New_Agent_Specialised Paradigms,Match_Specialised Paradigms,overall_validated_reasoning,audit_commentary,confidence,validated_bibliography_count,validated_bibliography_lines,Match_All_5
0,2018 - 8,gpt-4o-mini,Mixed,Mixed,MATCH,supervised,supervised,MATCH,classification; regression,classification; regression,...,MATCH,[],[],MATCH,The foundational paradigm is classified as Mix...,The study does not provide evidence of the app...,0.8,8,"[[1] page=4 | section=3. Methodology, quot...",MATCH
1,2019 - 33,gpt-4o-mini,Classical Statistical Modelling,Classical Statistical Modelling,MATCH,[],[],MATCH,[],[],...,MATCH,[],[],MATCH,The study employs conventional statistical tes...,The study's methodology does not support any s...,0.9,3,"[[1] page=2 | section=Methodology, quote: ...",MATCH
2,2020 - 8,gpt-4o-mini,Machine Learning,Machine Learning,MATCH,unsupervised,unsupervised,MATCH,clustering,clustering,...,MATCH,Natural Language Processing; Network Analysis,Natural Language Processing; Network Analysis,MATCH,The study applies machine learning topic model...,,1.0,7,"[[1] page=6 | section=JHTT, quote: We then...",MATCH
3,2021 - 28,gpt-4o-mini,Mixed,Mixed,MATCH,supervised,supervised,MATCH,regression,regression,...,MATCH,[],[],MATCH,The foundational paradigm is classified as 'Mi...,The classifications across layers are logicall...,0.9,7,"[[1] page=4 | section=4. Methods, quote: w...",MATCH
4,2021 - 56,gpt-4o-mini,Mixed,Mixed,MATCH,supervised; unsupervised,supervised; unsupervised,MATCH,clustering; regression,clustering,...,MATCH,Natural Language Processing; Time-Series Analy...,Natural Language Processing; Time-Series Analy...,MATCH,The study employs both supervised learning (XG...,The foundational paradigm was correctly identi...,1.0,14,"[[1] page=1 | section=Abstract, quote: Thi...",MISMATCH
5,2023 - 58,gpt-4o-mini,Machine Learning,Machine Learning,MATCH,unsupervised,unsupervised,MATCH,clustering,clustering,...,MATCH,Emerging Paradigms; Natural Language Processin...,Emerging Paradigms; Natural Language Processin...,MATCH,The study employs Latent Dirichlet Allocation ...,,1.0,13,[[1] page=6 | section=3.2. Topic modelling (LD...,MATCH
6,2024 - 99,gpt-4o-mini,Machine Learning,Machine Learning,MATCH,optimization-based learning,optimization-based learning,MATCH,reward optimisation,reward optimisation,...,MATCH,[],[],MATCH,The study employs a decision-analytic model th...,No specialised paradigms were found applicable...,0.9,3,[[1] page=5 | section=Optimization Preservatio...,MATCH


In [14]:
# Optional: inspect detailed reasoning for one rerun paper
if "full_retest" in globals() and len(full_retest):
    example_pid = "2021 - 28"
    show_iteration_reasoning(full_retest, example_pid, show_bibliography=True)


PAPER: 2021 - 28 | MODEL: gpt-4o-mini

Foundational Paradigm
  GT    : Mixed
  Agent : Mixed
  Match : MATCH

ML Learning Type
  GT    : supervised
  Agent : supervised
  Match : MATCH

ML Problem Type
  GT    : regression
  Agent : []
  Match : MISMATCH

Deep Learning Used
  GT    : False
  Agent : False
  Match : MATCH

Specialised Paradigms
  GT    : []
  Agent : []
  Match : MATCH

Overall all-5-dimensions match: MISMATCH
Confidence: 0.9
Overall validated reasoning:
  The foundational paradigm is classified as 'Mixed' due to the use of both classical statistical methods (Fama-
  MacBeth regressions) and machine learning methods (Elastic Net). The machine learning type is 'Supervised
  Learning' based on the application of Elastic Net for feature importance assessment. However, the problem type
  is classified as empty because the evidence does not specify a clear machine learning problem. The absence of
  specialised paradigms is validated, as no evidence supports the use of time-

In [15]:
run_modelling_iteration(
    paper_ids= ['2021 - 28'], # Enter Paper ID HERE. 
    llm_model="gpt-4o-mini",
    verbose=True,
    include_reasoning=True,
    show_reasoning_in_logs=True,
    log_bibliography=True
)

Re-running modelling node for 2021 - 28 with gpt-4o-mini...
  Assigned labels -> Foundational Paradigm: Mixed | ML Learning Type: supervised | ML Problem Type: regression | Deep Learning Used: False | Specialised Paradigms: []
  Match_All_5: MATCH

PAPER: 2021 - 28 | MODEL: gpt-4o-mini

Foundational Paradigm
  GT    : Mixed
  Agent : Mixed
  Match : MATCH

ML Learning Type
  GT    : supervised
  Agent : supervised
  Match : MATCH

ML Problem Type
  GT    : regression
  Agent : regression
  Match : MATCH

Deep Learning Used
  GT    : False
  Agent : False
  Match : MATCH

Specialised Paradigms
  GT    : []
  Agent : []
  Match : MATCH

Overall all-5-dimensions match: MATCH
Confidence: 0.9
Overall validated reasoning:
  The study employs both machine learning and classical statistical methods. Specifically, it uses Elastic Net,
  a machine learning tool aimed at feature selection and prediction, indicating a predictive modelling goal [1].
  Additionally, it examines the statistical impor

,Paper ID,Model,GT_Foundational Paradigm,New_Agent_Foundational Paradigm,Match_Foundational Paradigm,GT_ML Learning Type,New_Agent_ML Learning Type,Match_ML Learning Type,GT_ML Problem Type,New_Agent_ML Problem Type,...,Match_Deep Learning Used,GT_Specialised Paradigms,New_Agent_Specialised Paradigms,Match_Specialised Paradigms,overall_validated_reasoning,audit_commentary,confidence,validated_bibliography_count,validated_bibliography_lines,Match_All_5
0,2021 - 28,gpt-4o-mini,Mixed,Mixed,MATCH,supervised,supervised,MATCH,regression,regression,...,MATCH,[],[],MATCH,The study employs both machine learning and cl...,No specialised paradigms were found in the stu...,0.9,4,"[[1] page=4 | section=4. Methods, quote: w...",MATCH


# Open AI Direct PDF Upload

In [8]:
paper_path = r"C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\DATA_sample_10\2023 - 58.pdf"

In [9]:
from openai import OpenAI
client = OpenAI()

file = client.files.create(
    file=open(paper_path, "rb"),
    purpose="assistants"   # important
)

In [11]:
vector_store = client.vector_stores.create(
    name="papers_store"
)

In [12]:
client.vector_stores.files.create(
    vector_store_id=vector_store.id,
    file_id=file.id
)

VectorStoreFile(id='file-Ws7WK7ZM3Mhi7VbZUWybCJ', created_at=1775281774, last_error=None, object='vector_store.file', status='in_progress', usage_bytes=0, vector_store_id='vs_69d0a66be3f8819186f367a166a7ddbf', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))

In [18]:
prompt_path = r"C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\prompts\dsrp\modelling\foundational\foundational_classifier.yaml"

In [19]:
import yaml

with open(prompt_path, "r") as f:
    config = yaml.safe_load(f)

In [20]:
system_prompt = config["instructions"]   # this is your full classifier logic
temperature = config.get("temperature", 0)

In [23]:
response = client.responses.create(
    model="gpt-4o-mini",
    temperature=0,
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store.id]
    }],
    input=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": "Analyse the paper and return the classification. Focus on methodology. Return ONLY JSON."
        }
    ]
)

In [24]:
print(response.output_text)

```json
{
  "foundational_paradigm": "Mixed",
  "confidence": 0.9,
  "reasoning_explanation": "The study employs both classical statistical methods and machine learning techniques. It utilizes Latent Dirichlet Allocation (LDA) for topic modeling, which is a machine learning approach, alongside visual content analysis (VCA) and spatial analysis, which are more traditional statistical methods. The combination of these methodologies indicates a mixed approach, as the study aims to leverage the interpretability of statistical models while also utilizing the predictive capabilities of machine learning.",
  "bibliography": []
}
```


In [28]:
response

Response(id='resp_0aa2331526ed4ddb0069d0a6c7e4988197a677afee7128de74', created_at=1775281864.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseFileSearchToolCall(id='fs_0aa2331526ed4ddb0069d0a6c8ee6081978e957cb37b3dd2df', queries=['methodology', 'research design', 'statistical analysis', 'machine learning', 'modeling approach'], status='completed', type='file_search_call', results=None), ResponseOutputMessage(id='msg_0aa2331526ed4ddb0069d0a6ce2a848197b5d5678492a81160', content=[ResponseOutputText(annotations=[AnnotationFileCitation(file_id='file-Ws7WK7ZM3Mhi7VbZUWybCJ', filename='2023 - 58.pdf', index=603, type='file_citation'), AnnotationFileCitation(file_id='file-Ws7WK7ZM3Mhi7VbZUWybCJ', filename='2023 - 58.pdf', index=603, type='file_citation')], text='```json\n{\n  "foundational_paradigm": "Mixed",\n  "confidence": 0.9,\n  "reasoning_explanation": "The study employs both classical statistical m

In [29]:
from openai import OpenAI

client = OpenAI()

user_query = system_prompt

results = client.vector_stores.search(
    vector_store_id=vector_store.id,
    query=user_query,
)

In [35]:
def format_results(results):
    formatted_results = ''
    for result in results.data:
        formatted_result = f"<result file_id='{result.file_id}' file_name='{result.file_name}'>"
        for part in result.content:
            formatted_result += f"<content>{part.text}</content>"
        formatted_results += formatted_result + "</result>"
    return f"<sources>{formatted_results}</sources>"

In [36]:
formatted_results = format_results(results.data)

'\n'.join('\n'.join(c.text) for c in result.content for result in results.data)

completion = client.chat.completions.create(
    model="gpt-4.1",
    messages=[
        {
            "role": "developer",
            "content": "Produce a concise answer to the query based on the provided sources."
        },
        {
            "role": "user",
            "content": f"Sources: {formatted_results}\n\nQuery: '{user_query}'"
        }
    ],
)

print(completion.choices[0].message.content)

AttributeError: 'list' object has no attribute 'data'

In [37]:
client.vector_stores.list()

SyncCursorPage[VectorStore](data=[VectorStore(id='vs_69d0a66be3f8819186f367a166a7ddbf', created_at=1775281772, file_counts=FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1), last_active_at=1775282152, metadata={}, name='papers_store', object='vector_store', status='completed', usage_bytes=123983, expires_after=None, expires_at=None, description=None), VectorStore(id='vs_69d09b560d688191854c9a17129b4b42', created_at=1775278934, file_counts=FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1), last_active_at=1775278936, metadata={}, name='papers_store', object='vector_store', status='completed', usage_bytes=123983, expires_after=None, expires_at=None, description=None), VectorStore(id='vs_69d09b20475481919e3c5d99dcd160e9', created_at=1775278880, file_counts=FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1), last_active_at=1775278881, metadata={}, name='papers_store', object='vector_store', status='completed', usage_bytes=123983,

# Run using Open AI Vector Store

In [38]:
from modules.tools import add_files_to_openai_vector_store

result = add_files_to_openai_vector_store(
    file_paths=[
        "my_papers/paper1.pdf",
        "my_papers/paper2.pdf",
    ],
    vector_store_name="phd-methodology-store"
)
print(result)

OpenAIVectorStoreError: File not found: my_papers/paper1.pdf